In [1]:
#============================================================
# Celda 1 — Setup y rutas
#============================================================
import pandas as pd
import numpy as np
from pathlib import Path
import os
os.chdir("/workspaces/TFG_Spain-s_Migratory_Flow")

SETID   = Path("data/conectividad/setid")
RAW_DIR = Path("output/conectividad/01-raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

F_OLD = SETID / "Cobertura_BA_España_2013-2020_ESP_MUN_PROV_CCAA.xlsx"
F_NEW = SETID / "Cobertura_BA_España_2021-2024_MUN_PROV_CCAA.xlsx"

assert F_OLD.exists(), f"❌ No encontrado: {F_OLD}"
assert F_NEW.exists(), f"❌ No encontrado: {F_NEW}"
print("✅ Archivos encontrados")
print(f"   {F_OLD.name}")
print(f"   {F_NEW.name}")

✅ Archivos encontrados
   Cobertura_BA_España_2013-2020_ESP_MUN_PROV_CCAA.xlsx
   Cobertura_BA_España_2021-2024_MUN_PROV_CCAA.xlsx


In [2]:
#============================================================
# Celda 2 — Excel 2013-2020 | hoja Provincia → RAW parquet
#============================================================
df_old = pd.read_excel(F_OLD, sheet_name="Provincia")

# Columnas Cob. 100Mbps + identifiers
ID_COLS = ["Comunidad Autónoma", "Provincia"]
cols_100 = [c for c in df_old.columns if "Cob. 100Mbps" in c]

print(f"Columnas 100Mbps encontradas ({len(cols_100)}):")
for c in cols_100:
    print(f"  {repr(c)}")

df_old_raw = df_old[ID_COLS + cols_100].copy()
df_old_raw.to_parquet(RAW_DIR / "raw_cobertura_2013_2020.parquet", index=False)
print(f"\n✅ Guardado: raw_cobertura_2013_2020.parquet — shape: {df_old_raw.shape}")

Columnas 100Mbps encontradas (8):
  'Cob. 100Mbps\n(dic 2013)'
  'Cob. 100Mbps\n(dic 2014)'
  'Cob. 100Mbps\n(dic 2015)'
  'Cob. 100Mbps\n(junio 2016)'
  'Cob. 100Mbps\n(junio 2017)'
  'Cob. 100Mbps\n(junio 2018)'
  'Cob. 100Mbps\n(junio 2019)'
  'Cob. 100Mbps\n(junio 2020)'

✅ Guardado: raw_cobertura_2013_2020.parquet — shape: (52, 10)


In [3]:
#============================================================
# Celda 3 — Excel 2021-2024 | hoja Provincia_%viviendas → RAW parquet
#============================================================
df_new = pd.read_excel(F_NEW, sheet_name="Provincia_%viviendas")

ID_COLS = ["Comunidad Autónoma", "Provincia"]
cols_100 = [c for c in df_new.columns if "100Mbps" in c]

print(f"Columnas 100Mbps encontradas ({len(cols_100)}):")
for c in cols_100:
    print(f"  {repr(c)}")

df_new_raw = df_new[ID_COLS + cols_100].copy()
df_new_raw.to_parquet(RAW_DIR / "raw_cobertura_2021_2024.parquet", index=False)
print(f"\n✅ Guardado: raw_cobertura_2021_2024.parquet — shape: {df_new_raw.shape}")

Columnas 100Mbps encontradas (4):
  'Cob. 100Mbps\n(junio 2021)'
  'Cob. 100Mbps condiciones máxima demanda\n(junio 2022)'
  'Cob. 100Mbps condiciones máxima demanda\n(junio 2023)'
  'Cob. 100Mbps condiciones máxima demanda\n(junio 2024)'

✅ Guardado: raw_cobertura_2021_2024.parquet — shape: (52, 6)


In [4]:
#============================================================
# Celda 4 — Verificación: listar parquets generados
#============================================================
parquets = sorted(RAW_DIR.glob("*.parquet"))
print(f"📂 Parquets en {RAW_DIR}:")
for p in parquets:
    df = pd.read_parquet(p)
    print(f"   {p.name} → shape: {df.shape}")
    print(f"   Columnas: {list(df.columns)}")
    print()

📂 Parquets en output/conectividad/01-raw:
   raw_cobertura_2013_2020.parquet → shape: (52, 10)
   Columnas: ['Comunidad Autónoma', 'Provincia', 'Cob. 100Mbps\n(dic 2013)', 'Cob. 100Mbps\n(dic 2014)', 'Cob. 100Mbps\n(dic 2015)', 'Cob. 100Mbps\n(junio 2016)', 'Cob. 100Mbps\n(junio 2017)', 'Cob. 100Mbps\n(junio 2018)', 'Cob. 100Mbps\n(junio 2019)', 'Cob. 100Mbps\n(junio 2020)']

   raw_cobertura_2021_2024.parquet → shape: (52, 6)
   Columnas: ['Comunidad Autónoma', 'Provincia', 'Cob. 100Mbps\n(junio 2021)', 'Cob. 100Mbps condiciones máxima demanda\n(junio 2022)', 'Cob. 100Mbps condiciones máxima demanda\n(junio 2023)', 'Cob. 100Mbps condiciones máxima demanda\n(junio 2024)']

   raw_metadatos_conectividad.parquet → shape: (2, 9)
   Columnas: ['fuente', 'url_datos', 'frecuencia', 'ultimo_informe', 'tipo_datos', 'granularidad', 'estado', 'nota', 'timestamp']

